In [1]:
import pandas as pd
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm  

/home/zman/anaconda3/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read the Excel file
excel_path = "/media/zman/extrahd/reu20024project/List of Analysis Concepts to check for Duplicates.xlsx"
sheet_names = pd.ExcelFile(excel_path).sheet_names

# Load each sheet into a separate dataframe
df_sheet1 = pd.read_excel(excel_path, sheet_name=sheet_names[0])
df_sheet2 = pd.read_excel(excel_path, sheet_name=sheet_names[1])
df_sheet3 = pd.read_excel(excel_path, sheet_name=sheet_names[2])

# Print the names of the sheets to verify
print("Sheet names:", sheet_names)

Sheet names: ['Analysis Concepts', 'OIFs and Supplements', 'Sheet1']


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:


# Initialize the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-distilroberta-v1', clean_up_tokenization_spaces=True)
model = AutoModel.from_pretrained('sentence-transformers/all-distilroberta-v1')
model.to(device)


def chunk_text(text, max_tokens=512):
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk = tokenizer.decode(tokens[i:i+max_tokens])
        chunks.append(chunk)
    return chunks

def get_embedding_safe(text):
    if isinstance(text, str):
        text = text.replace("\n", " ")
        try:
            #text_chunks = chunk_text(text, max_tokens=512)
            embeddings = []
            #print(text_chunks)
            #for chunk in text_chunks:
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)

            with torch.no_grad():
                outputs = model(**inputs)
            #embedding = outputs.pooler_output[0].cpu().numpy() 
            embedding = outputs.last_hidden_state.mean(dim=1)[0].cpu().numpy() 
                #embedding = outputs.pooler_output[0].numpy()  
            embeddings.append(embedding)
            return embeddings[0]
            #return np.mean(embeddings, axis=0)
        except Exception as e:
            print(f"An error occurred: {e}")
            return None
    else:
        return None
    
# Function to get query embedding
def get_query_embedding(question):
    print("Received question:", question)  # Debug: question input
    inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
    question_embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()

    print("Question embedding shape:", question_embedding.shape)  # Debug: shape of question embedding
    return question_embedding


# Function to compute cosine similarity
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Query and retrieval function
def retrieve_closest_text(question, embeddings_df, text_df, section="Title"):
    question_embedding = get_query_embedding(question)
    
    # Calculate cosine similarity for each embedding in the specified section
    def calculate_similarity(x):
       
        try:
            
            page_embedding = x
            
            # Check if either embedding is zero to avoid -inf in cosine similarity
            if np.linalg.norm(question_embedding) == 0 or np.linalg.norm(page_embedding) == 0:
                print("Warning: Zero vector detected in embeddings.")
                return -3333
            
            # Calculate cosine similarity
            return cosine_similarity(question_embedding, page_embedding)
        except Exception as e:
            print(f"Error evaluating embedding: {e}")
            return -222
       

    similarities = embeddings_df[section].apply(calculate_similarity)
    print("Similarity scores:", similarities.head())  # Debug: check similarity scores

    # Get indices of the top 5 highest similarities (use nlargest for most similar results)
    top_5_indices = similarities.nlargest(5).index
    top_5_texts = text_df.loc[top_5_indices, section]
    
    print(f"Top 5 matches in '{section}' section:")
    for i, text in enumerate(top_5_texts):
        print(f"Match {i+1}: {text[:500]}")  # Show the first 500 characters for readability
    
    # Return the closest match
    closest_idx = top_5_indices[0]
    closest_text = text_df.loc[closest_idx, section]
    
    return closest_text



In [5]:
df_sheet1.columns

Index(['ID', 'EC Identifier_Analysis Proposal',
       'ECHO Cohort Name and Award # or ECHO Component', 'Moved to Proposal',
       'Writing Team Leader', 'Title', 'Hypothesis', 'Objectives',
       'Primary Exposures', 'ECHO Groups(s) of Interest', 'Open Date',
       'Close Date', 'Use of Biospecimens', 'Comments on Biospecimens',
       'Contact Method', 'Biospecimen Type',
       'Areas of Exposure & Response Measure',
       'Writing Team Leader Institution', 'Phone', 'Email', 'Discussion Alert',
       'Co-Author(s)', 'Co-Author(s) Institution(s)', 'Co-Author(s) Email',
       'Created', 'Created By', 'Type of Analysis', 'BioSpecimen Type Other',
       'Groups Other', 'Analysis use of data', 'OtherAnalysisUseOfData',
       'Who analyze data', 'Otherwhoanalyzedata', 'Discussion short video',
       'Step 1 Concept Form Workflow', 'Step 1 Concept Form Workflow2',
       'Item Type', 'Path'],
      dtype='object')

In [6]:
# Function to get query embedding
def get_query_embedding(question):
    print("Received question:", question)  # Debug: question input
    inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
    question_embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()

    print("Question embedding shape:", question_embedding)  # Debug: shape of question embedding
    return question_embedding


get_query_embedding("How are you?")

Received question: How are you?
Question embedding shape: [-2.12793514e-01  1.68231875e-01  3.42981637e-01  3.55151057e-01
  9.05264854e-01 -6.53780401e-01  5.51170290e-01  3.14009249e-01
  4.18947577e-01  4.57741350e-01 -3.19651924e-02  2.42664695e-01
 -1.62727591e-02  5.17540038e-01 -5.09953380e-01  5.23107648e-01
  5.12095213e-01 -5.92671633e-01 -1.77316070e-01  1.74069345e-01
  5.65181613e-01 -2.43221998e-01  4.68647599e-01  4.18245643e-02
  6.45657778e-01  2.50956446e-01  4.89922613e-01 -4.58356380e-01
  1.76401541e-01  5.99299632e-02 -4.38891053e-01 -5.09204805e-01
 -3.39616954e-01 -1.59562677e-01 -1.37271255e-01  1.90929055e-01
  3.80916119e-01 -5.02894223e-02  1.26079321e-01  6.14610970e-01
 -3.37296098e-01 -5.10724306e-01  6.15460515e-01  7.23140180e-01
 -2.00564414e-01 -1.33041322e-01  2.83766866e-01 -4.89111632e-01
 -2.75532424e-01  8.35284948e-01  3.79122019e-01 -5.87579787e-01
  3.25263917e-01 -2.88623452e-01  2.59730935e-01 -4.28385258e-01
 -3.88097972e-01  1.74908102e-01

array([-2.12793514e-01,  1.68231875e-01,  3.42981637e-01,  3.55151057e-01,
        9.05264854e-01, -6.53780401e-01,  5.51170290e-01,  3.14009249e-01,
        4.18947577e-01,  4.57741350e-01, -3.19651924e-02,  2.42664695e-01,
       -1.62727591e-02,  5.17540038e-01, -5.09953380e-01,  5.23107648e-01,
        5.12095213e-01, -5.92671633e-01, -1.77316070e-01,  1.74069345e-01,
        5.65181613e-01, -2.43221998e-01,  4.68647599e-01,  4.18245643e-02,
        6.45657778e-01,  2.50956446e-01,  4.89922613e-01, -4.58356380e-01,
        1.76401541e-01,  5.99299632e-02, -4.38891053e-01, -5.09204805e-01,
       -3.39616954e-01, -1.59562677e-01, -1.37271255e-01,  1.90929055e-01,
        3.80916119e-01, -5.02894223e-02,  1.26079321e-01,  6.14610970e-01,
       -3.37296098e-01, -5.10724306e-01,  6.15460515e-01,  7.23140180e-01,
       -2.00564414e-01, -1.33041322e-01,  2.83766866e-01, -4.89111632e-01,
       -2.75532424e-01,  8.35284948e-01,  3.79122019e-01, -5.87579787e-01,
        3.25263917e-01, -

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [8]:
columns_to_process = ["Title", "Hypothesis", "Objectives", "Primary Exposures"]

embeddings_dict = {col: [] for col in columns_to_process}

print("Starting embedding computation...")
df = df_sheet1

for idx, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing Rows"):
    for col in columns_to_process:
        text = row[col]
        embedding = get_embedding_safe(text)
        embeddings_dict[col].append(embedding)

Starting embedding computation...


Processing Rows: 100%|██████████| 623/623 [00:09<00:00, 66.59it/s]


In [9]:
print("Creating embeddings DataFrame...")
embeddings_df = pd.DataFrame(embeddings_dict)

print("Converting embeddings to lists...")

for col in columns_to_process:
    embeddings_df[col] = embeddings_df[col].apply(lambda emb: emb.tolist() if isinstance(emb, np.ndarray) else emb)

Creating embeddings DataFrame...
Converting embeddings to lists...


In [ ]:
# Read the CSV files
text_df = df_sheet1

# Example usage
question1 = "What concepts are studied related to sleep health?"

closest_abstract = retrieve_closest_text(question1, embeddings_df, text_df, section="Hypothesis")


Received question: What conce[ts are studied related to sleep health?
Question embedding shape: [-2.15595201e-01 -1.02595955e-01  2.00879410e-01 -1.94483504e-01
 -1.68217689e-01  1.37683108e-01 -5.42961471e-02 -1.34037539e-01
 -9.05812010e-02 -1.30582765e-01 -1.80639416e-01 -4.02976334e-01
 -1.66865095e-01 -8.84054229e-02 -1.35867000e-01 -7.43070364e-01
  8.71554494e-01 -7.11588040e-02 -1.45182714e-01 -2.24746570e-01
  4.07855302e-01  8.14175606e-02 -6.55224025e-01 -1.58273354e-01
  4.27305728e-01  5.62215626e-01 -7.66502798e-01  1.05405673e-01
  5.47224522e-01 -5.25561683e-02 -3.30753386e-01  5.21010101e-01
 -1.84264734e-01 -1.86235726e-01 -1.04984730e-01 -1.25344144e-02
 -1.27238736e-01 -3.40349644e-01  7.89413989e-01  3.36813539e-01
 -8.22598264e-02  6.63552403e-01  7.35743344e-02  3.69719625e-01
  3.81388754e-01  1.66932583e-01  1.15174368e-01  6.49079233e-02
 -2.23300949e-01 -1.61911041e-01  4.67054158e-01 -6.01614475e-01
  2.65172094e-01 -1.70693517e-01 -4.93390381e-01 -1.4510138

In [11]:
import os
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

max_seq_length = 512


# 2. Load Llama3 model
model, tokenizer = FastLanguageModel.from_pretrained(
    #model_name = "unsloth/llama-3-70b-bnb-4bit",
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
    cache_dir='/media/zman/extrahd/reu20024project/',
   
    device_map="auto"
)

model = FastLanguageModel.for_inference(model)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.10.5: Fast Llama patching. Transformers = 4.45.2.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.475 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.5.0. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post2. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


Unsloth: We fixed a gradient accumulation bug, but it seems like you don't have the latest transformers version!
Please update transformers, TRL and unsloth via:
`pip install --upgrade --no-cache-dir unsloth git+https://github.com/huggingface/transformers.git git+https://github.com/huggingface/trl.git`


In [12]:
# 3 Before training
def generate_text(text, model):
    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")
    outputs = model.generate(**inputs, max_new_tokens=1028)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [13]:
prompt = "Answer the question at the end with only the information provided in the context. Context: " + closest_abstract + ". " + question1
prompt

'Answer the question at the end with only the information provided in the context. Context: We will use ECHO-wide longitudinal data via the ECHO platform to test 3 groups of hypotheses related to social and environmental influences on sleep health trajectories across childhood. These include the following: HYPOTHESIS 1:  1a) After controlling for potential confounds (e.g. socioeconomic status, sex, race and ethnicity), food insecurity will be associated with concurrent sleep health; 2b) After controlling for potential confounds (e.g. socioeconomic status, sex, race and ethnicity), food insecurity will be predictive of future sleep health.  Rationale 1:  Food insecurity (i.e., lack of consistent access to adequate, nutritious foods) is a leading health crisis in the US. Although a large body of literature indicates links between income & sleep outcomes, few, if any, studies have examined the relationship between food insecurity and sleep, especially during childhood. HYPOTHESIS 2:  2a) 

In [14]:
response = generate_text(prompt, model)

In [15]:
import textwrap

def print_wrapped_text(text, width=120):
    """
    Prints the given text with word wrapping to the specified width.

    Parameters:
    text (str): The text to be printed.
    width (int): The maximum width of each line (default is 120 characters).
    """
    wrapped_text = textwrap.fill(text, width=width)
    print(wrapped_text)

In [16]:
print_wrapped_text(response)

Answer the question at the end with only the information provided in the context. Context: We will use ECHO-wide
longitudinal data via the ECHO platform to test 3 groups of hypotheses related to social and environmental influences on
sleep health trajectories across childhood. These include the following: HYPOTHESIS 1:  1a) After controlling for
potential confounds (e.g. socioeconomic status, sex, race and ethnicity), food insecurity will be associated with
concurrent sleep health; 2b) After controlling for potential confounds (e.g. socioeconomic status, sex, race and
ethnicity), food insecurity will be predictive of future sleep health.  Rationale 1:  Food insecurity (i.e., lack of
consistent access to adequate, nutritious foods) is a leading health crisis in the US. Although a large body of
literature indicates links between income & sleep outcomes, few, if any, studies have examined the relationship between
food insecurity and sleep, especially during childhood. HYPOTHESIS 2:  2a) C